## 01. IMPORTS

In [1]:
from pathlib import Path
import json
import joblib

import numpy as np
import pandas as pd

from sklearn.dummy import (
    DummyRegressor,
    DummyClassifier
)

from sklearn.linear_model import (
    Ridge,
    LogisticRegression
)

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)


print("Baseline modelling libraries imported successfully.")

Baseline modelling libraries imported successfully.


## 02. FILE LOCATIONS

In [2]:
PROJECT_ROOT = Path(
    r"C:\Users\bongo\OneDrive\Desktop\GitHub\Movie Analysis Predictor"
)

MODEL_READY_DIR = (
    PROJECT_ROOT
    / "Data"
    / "Model_Ready"
)

FEATURE_DIR = (
    MODEL_READY_DIR
    / "Features"
)

TARGET_DIR = (
    MODEL_READY_DIR
    / "Targets"
)

METADATA_DIR = (
    MODEL_READY_DIR
    / "Metadata"
)

MODEL_OUTPUT_DIR = (
    PROJECT_ROOT
    / "Models"
    / "Baseline"
)

RESULTS_OUTPUT_DIR = (
    PROJECT_ROOT
    / "Data"
    / "Model_Results"
    / "Baseline"
)

MODEL_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

RESULTS_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Baseline model output:")
print(MODEL_OUTPUT_DIR)

print("\nBaseline results output:")
print(RESULTS_OUTPUT_DIR)

Baseline model output:
C:\Users\bongo\OneDrive\Desktop\GitHub\Movie Analysis Predictor\Models\Baseline

Baseline results output:
C:\Users\bongo\OneDrive\Desktop\GitHub\Movie Analysis Predictor\Data\Model_Results\Baseline


## 03. LOAD MODEL-READY DATA

In [3]:
X_train = pd.read_csv(
    FEATURE_DIR / "X_train.csv"
)

X_validation = pd.read_csv(
    FEATURE_DIR / "X_validation.csv"
)

X_test = pd.read_csv(
    FEATURE_DIR / "X_test.csv"
)


y_train_reg = pd.read_csv(
    TARGET_DIR / "y_train_regression.csv"
)["log_worldwide_box_office"]

y_validation_reg = pd.read_csv(
    TARGET_DIR / "y_validation_regression.csv"
)["log_worldwide_box_office"]

y_test_reg = pd.read_csv(
    TARGET_DIR / "y_test_regression.csv"
)["log_worldwide_box_office"]


y_train_cls = pd.read_csv(
    TARGET_DIR / "y_train_classification.csv"
)["blockbuster"]

y_validation_cls = pd.read_csv(
    TARGET_DIR / "y_validation_classification.csv"
)["blockbuster"]

y_test_cls = pd.read_csv(
    TARGET_DIR / "y_test_classification.csv"
)["blockbuster"]


print("=" * 80)
print("BASELINE MODELLING DATA")
print("=" * 80)

print("\nFeatures:")
print("Train:     ", X_train.shape)
print("Validation:", X_validation.shape)
print("Test:      ", X_test.shape)

print("\nRegression targets:")
print("Train:     ", len(y_train_reg))
print("Validation:", len(y_validation_reg))
print("Test:      ", len(y_test_reg))

print("\nClassification targets:")
print("Train:     ", len(y_train_cls))
print("Validation:", len(y_validation_cls))
print("Test:      ", len(y_test_cls))

BASELINE MODELLING DATA

Features:
Train:      (6668, 36)
Validation: (600, 36)
Test:       (211, 36)

Regression targets:
Train:      6668
Validation: 600
Test:       211

Classification targets:
Train:      6668
Validation: 600
Test:       211


## 04. REGRESSION METRIC HELPER

In [4]:
def evaluate_regression(
    model_name,
    y_true_log,
    y_pred_log
):

    # --------------------------------------------------------
    # Log-scale metrics
    # --------------------------------------------------------

    mae_log = mean_absolute_error(
        y_true_log,
        y_pred_log
    )

    rmse_log = np.sqrt(
        mean_squared_error(
            y_true_log,
            y_pred_log
        )
    )

    r2 = r2_score(
        y_true_log,
        y_pred_log
    )


    # --------------------------------------------------------
    # Convert back to dollar values
    # --------------------------------------------------------

    y_true_usd = np.expm1(
        y_true_log
    )

    y_pred_usd = np.expm1(
        y_pred_log
    )

    # Prevent impossible negative revenue predictions
    y_pred_usd = np.maximum(
        y_pred_usd,
        0
    )


    mae_usd = mean_absolute_error(
        y_true_usd,
        y_pred_usd
    )

    rmse_usd = np.sqrt(
        mean_squared_error(
            y_true_usd,
            y_pred_usd
        )
    )


    return {
        "model": model_name,
        "mae_log": mae_log,
        "rmse_log": rmse_log,
        "r2": r2,
        "mae_usd": mae_usd,
        "rmse_usd": rmse_usd
    }


print(
    "Regression evaluation function created."
)

Regression evaluation function created.


## 05. DUMMY REGRESSION BASELINE

In [5]:
dummy_regressor = DummyRegressor(
    strategy="median"
)

dummy_regressor.fit(
    X_train,
    y_train_reg
)

dummy_reg_validation_pred = (
    dummy_regressor.predict(
        X_validation
    )
)

dummy_reg_results = (
    evaluate_regression(
        "Dummy Regressor",
        y_validation_reg,
        dummy_reg_validation_pred
    )
)

print("=" * 80)
print("DUMMY REGRESSION BASELINE")
print("=" * 80)

display(
    pd.DataFrame(
        [dummy_reg_results]
    )
)

DUMMY REGRESSION BASELINE


,model,mae_log,rmse_log,r2,mae_usd,rmse_usd
0,Dummy Regressor,1.909764,2.37789,-0.024839,1.335019e+08,2.789626e+08


## 06. RIDGE REGRESSION BASELINE

In [6]:
ridge_baseline = Ridge(
    alpha=1.0
)

ridge_baseline.fit(
    X_train,
    y_train_reg
)

ridge_validation_pred = (
    ridge_baseline.predict(
        X_validation
    )
)

ridge_results = (
    evaluate_regression(
        "Ridge Regression",
        y_validation_reg,
        ridge_validation_pred
    )
)


regression_baseline_results = (
    pd.DataFrame([
        dummy_reg_results,
        ridge_results
    ])
)

print("=" * 80)
print("REGRESSION BASELINE COMPARISON")
print("=" * 80)

display(
    regression_baseline_results
)

REGRESSION BASELINE COMPARISON


,model,mae_log,rmse_log,r2,mae_usd,rmse_usd
0,Dummy Regressor,1.909764,2.37789,-0.024839,1.335019e+08,2.789626e+08
1,Ridge Regression,1.135776,1.55364,0.562505,2.253186e+08,1.611513e+09


## 07. CLASSIFICATION EVALUATION HELPER

In [7]:
def evaluate_classification(
    model_name,
    y_true,
    y_pred,
    y_probability
):

    return {
        "model": model_name,

        "accuracy":
            accuracy_score(
                y_true,
                y_pred
            ),

        "precision":
            precision_score(
                y_true,
                y_pred,
                zero_division=0
            ),

        "recall":
            recall_score(
                y_true,
                y_pred,
                zero_division=0
            ),

        "f1":
            f1_score(
                y_true,
                y_pred,
                zero_division=0
            ),

        "roc_auc":
            roc_auc_score(
                y_true,
                y_probability
            )
    }


print(
    "Classification evaluation function created."
)

Classification evaluation function created.


## 08. DUMMY CLASSIFICATION BASELINE

In [8]:
dummy_classifier = DummyClassifier(
    strategy="prior",
    random_state=42
)

dummy_classifier.fit(
    X_train,
    y_train_cls
)

dummy_cls_validation_pred = (
    dummy_classifier.predict(
        X_validation
    )
)

dummy_cls_validation_prob = (
    dummy_classifier.predict_proba(
        X_validation
    )[:, 1]
)


dummy_cls_results = (
    evaluate_classification(
        "Dummy Classifier",
        y_validation_cls,
        dummy_cls_validation_pred,
        dummy_cls_validation_prob
    )
)


print("=" * 80)
print("DUMMY CLASSIFICATION BASELINE")
print("=" * 80)

display(
    pd.DataFrame([
        dummy_cls_results
    ])
)

DUMMY CLASSIFICATION BASELINE


,model,accuracy,precision,recall,f1,roc_auc
0,Dummy Classifier,0.785,0.0,0.0,0.0,0.5


## 09. LOGISTIC REGRESSION BASELINE

In [10]:
logistic_baseline = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    random_state=42
)

logistic_baseline.fit(
    X_train,
    y_train_cls
)


logistic_validation_pred = (
    logistic_baseline.predict(
        X_validation
    )
)

logistic_validation_prob = (
    logistic_baseline.predict_proba(
        X_validation
    )[:, 1]
)


logistic_results = (
    evaluate_classification(
        "Logistic Regression",
        y_validation_cls,
        logistic_validation_pred,
        logistic_validation_prob
    )
)


classification_baseline_results = (
    pd.DataFrame([
        dummy_cls_results,
        logistic_results
    ])
)


print("=" * 80)
print("CLASSIFICATION BASELINE COMPARISON")
print("=" * 80)

display(
    classification_baseline_results
)

CLASSIFICATION BASELINE COMPARISON


,model,accuracy,precision,recall,f1,roc_auc
0,Dummy Classifier,0.785000,0.000000,0.000000,0.000000,0.50000
1,Logistic Regression,0.781667,0.495652,0.883721,0.635097,0.90673


## 10. LOGISTIC CONFUSION MATRIX

In [12]:
baseline_confusion_matrix = (
    confusion_matrix(
        y_validation_cls,
        logistic_validation_pred
    )
)

confusion_matrix_df = pd.DataFrame(
    baseline_confusion_matrix,
    index=[
        "Actual Non-Blockbuster",
        "Actual Blockbuster"
    ],
    columns=[
        "Predicted Non-Blockbuster",
        "Predicted Blockbuster"
    ]
)


print("=" * 80)
print("LOGISTIC REGRESSION CONFUSION MATRIX")
print("=" * 80)

display(
    confusion_matrix_df
)

LOGISTIC REGRESSION CONFUSION MATRIX


,Predicted Non-Blockbuster,Predicted Blockbuster
Actual Non-Blockbuster,355,116
Actual Blockbuster,15,114


## 11. SAVE BASELINE RESULTS

In [13]:
regression_baseline_results.to_csv(
    RESULTS_OUTPUT_DIR
    / "regression_baseline_results.csv",
    index=False
)

classification_baseline_results.to_csv(
    RESULTS_OUTPUT_DIR
    / "classification_baseline_results.csv",
    index=False
)

confusion_matrix_df.to_csv(
    RESULTS_OUTPUT_DIR
    / "logistic_regression_confusion_matrix.csv"
)


joblib.dump(
    dummy_regressor,
    MODEL_OUTPUT_DIR
    / "dummy_regressor.joblib"
)

joblib.dump(
    ridge_baseline,
    MODEL_OUTPUT_DIR
    / "ridge_regression_baseline.joblib"
)

joblib.dump(
    dummy_classifier,
    MODEL_OUTPUT_DIR
    / "dummy_classifier.joblib"
)

joblib.dump(
    logistic_baseline,
    MODEL_OUTPUT_DIR
    / "logistic_regression_baseline.joblib"
)


print("=" * 80)
print("BASELINE OUTPUTS SAVED")
print("=" * 80)

print("\nModels:")
print(MODEL_OUTPUT_DIR)

print("\nResults:")
print(RESULTS_OUTPUT_DIR)

BASELINE OUTPUTS SAVED

Models:
C:\Users\bongo\OneDrive\Desktop\GitHub\Movie Analysis Predictor\Models\Baseline

Results:
C:\Users\bongo\OneDrive\Desktop\GitHub\Movie Analysis Predictor\Data\Model_Results\Baseline


## 12. BASELINE MODELLING SUMMARY

In [14]:
baseline_summary = pd.DataFrame({

    "task": [
        "Regression",
        "Regression",
        "Classification",
        "Classification"
    ],

    "model": [
        "Dummy Regressor",
        "Ridge Regression",
        "Dummy Classifier",
        "Logistic Regression"
    ],

    "purpose": [
        "Naive regression benchmark",
        "Simple learned regression benchmark",
        "Naive classification benchmark",
        "Simple learned classification benchmark"
    ]
})


print("=" * 80)
print("BASELINE MODELLING — COMPLETE")
print("=" * 80)

display(
    baseline_summary
)

print(
    "\nThe test set remains untouched."
)

print(
    "\nNext deliverable: "
    "Main Model Development and Comparison."
)

BASELINE MODELLING — COMPLETE


,task,model,purpose
0,Regression,Dummy Regressor,Naive regression benchmark
1,Regression,Ridge Regression,Simple learned regression benchmark
2,Classification,Dummy Classifier,Naive classification benchmark
3,Classification,Logistic Regression,Simple learned classification benchmark



The test set remains untouched.

Next deliverable: Main Model Development and Comparison.
